In [6]:
import cv2 as cv
import numpy as np
import pandas as pd
from nmot import NMOT

In [7]:
def process_video(
    input_path: str,
    output_path: str = "tracked_output.mp4",
    csv_path: str = "tracks.csv",
    roi=None,
    dist2Threshold=50,
    knn_history = 100
):
    cap = cv.VideoCapture(input_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    if output_path is not None:
        fourcc = cv.VideoWriter_fourcc(*"mp4v")
        writer = cv.VideoWriter(output_path, fourcc, fps, (width, height))

    tracker = NMOT(
        warmup_frames=50,
        dist2Threshold=dist2Threshold,
        knn_history=knn_history,
        min_area=8,
        max_area=80,
        max_match_dist=25,
        max_missed=12,
        use_lk=True,
        roi=roi,
    )

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        vis, mask, active_tracks = tracker.update(frame)
        if output_path is not None: 
            writer.write(vis)

        
        # cv.imshow("tracks", vis)
        # cv.imshow("mask", mask)
        # if cv.waitKey(1) & 0xFF == 27:
        #     break

    cap.release()
    if output_path is not None: 
        writer.release()
    cv.destroyAllWindows()

    df = tracker.save_tracks(csv_path)

    return df

In [8]:
# import os
# from pathlib import Path
# from tqdm import tqdm
# dir_path = Path(r'E:\asp\Ants\P.yeensis')
# save_dir = Path(r'E:\asp\Ants\P.yeensis\nmot_tracks')

In [9]:
# pbar = tqdm(dir_path.glob("*.MP4"))
# for file_path in pbar:
#     csv_path = save_dir.joinpath(file_path.with_suffix('.csv').name)
#     output_path = save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
#     pbar.set_description(f"Processing {file_path.name}")
#     df = process_video(input_path=file_path,
#                        output_path=output_path,
#                        csv_path=csv_path,
#                        roi=None)
#     pbar.update(1)

In [10]:
import os
from pathlib import Path
from tqdm import tqdm
dir_path = Path(r'/home/akhiyarov/asp/NMOT/data/synth_gen_gt')
save_dir = Path(r'/home/akhiyarov/asp/NMOT/data/synth_gen_tracked')

In [11]:
# roi_dict = {'S1240004начало.MP4': (210, 70, 470, 340),
#             'S1240005.MP4': (210, 60, 490, 350),
#             'S1240006.MP4':(210, 60, 490, 360),
#             'S1240007.MP4':(210, 60, 490, 360),
#             'S1240008.MP4':(210, 60, 490, 360),
#             'S1240009.MP4':(210, 60, 490, 360),
#             'S1240010.MP4':(210, 60, 490, 360),
#             'S1240012.MP4':(210, 60, 490, 360),
#             'S1240013повороткреста.MP4':(210, 60, 490, 360),
#             'S1240014приманка.MP4':(210, 60, 490, 360),
#             'S1240015.MP4':(210, 60, 490, 360),
#             'S1240016.MP4':(210, 60, 490, 360),
#             'S1240017.MP4':(210, 60, 490, 360),
#             'S1240018.MP4':(210, 60, 490, 360),
#             'S1240019.MP4':(210, 60, 490, 360),
#             'S1240020.MP4':(210, 60, 490, 360),
#             'S1240021.MP4':(210, 60, 490, 360),
#             'S1240022.MP4':(210, 60, 490, 360),
#             'S1240023.MP4':(210, 60, 490, 360),
#             'S1240024.MP4':(210, 60, 490, 360),
#             'S1240025.MP4':(210, 60, 490, 360)}

In [12]:
# import pickle
# with open('frufa_roi.pkl', 'wb') as f:
#     pickle.dump(roi_dict, f)

In [13]:
# df = process_video(
#     input_path=r"/home/akhiyarov/asp/NMOT/data/gen_ants.mp4",
#     output_path="/home/akhiyarov/asp/NMOT/data/gen_ants_tracked.mp4",
#     csv_path="/home/akhiyarov/asp/NMOT/data/gen_ants_tracked.csv",
#     # roi=(210, 60, 490, 360),
#     dist2Threshold=200,
#     knn_history = 200
# )

# print(df.head())

In [14]:
pbar = tqdm(dir_path.glob("gen_ants_*.mp4"))
for file_path in pbar:
    csv_path = save_dir.joinpath(file_path.with_suffix('.csv').name)
    output_path = save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
    pbar.set_description(f"Processing {file_path.name}")
    df = process_video(input_path=file_path,
                       output_path=output_path,
                       csv_path=csv_path,
                    #    roi=roi_dict[file_path.name],
                       dist2Threshold=200,
                       knn_history = 200)
    pbar.update(1)

Processing gen_ants_3.mp4: : 20it [24:42, 74.13s/it] 
